In [ ]:
# Cell 1 – Install & Imports
!pip install -q torchvision torch tqdm scikit-learn matplotlib seaborn

import os, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import OxfordIIITPet

from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             classification_report)

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/model_checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Models will be saved to: {SAVE_DIR}')

: 

In [ ]:
# Cell 2 – Shared hyperparameters (keep these constant for fair comparison)
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 20          # increase to 20 if you have time/GPU
LR          = 1e-3
NUM_CLASSES = 37           # Oxford-IIIT Pet: 37 breeds

# Dataset split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

In [ ]:
# Cell 3 – Transforms (same for all experiments)
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],   # ImageNet mean/std
                         [0.229, 0.224, 0.225]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [ ]:
# Cell 4 – Download & create train / val / test splits
DATA_DIR = '/content/data'

# Download with train-split (trainval) for majority
full_trainval = OxfordIIITPet(DATA_DIR, split='trainval',
                               transform=None, download=True)
full_test_raw = OxfordIIITPet(DATA_DIR, split='test',
                               transform=None, download=True)

# Combine all indices for a clean manual split
full_ds = torch.utils.data.ConcatDataset([full_trainval, full_test_raw])
n = len(full_ds)
indices = list(range(n))
random.shuffle(indices)

n_train = int(TRAIN_RATIO * n)
n_val   = int(VAL_RATIO   * n)

train_idx = indices[:n_train]
val_idx   = indices[n_train : n_train + n_val]
test_idx  = indices[n_train + n_val:]

print(f'Total: {n}  |  Train: {len(train_idx)}  |  Val: {len(val_idx)}  |  Test: {len(test_idx)}')

# Apply transforms through wrapper
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, label = self.dataset[self.indices[i]]
        if self.transform:
            img = self.transform(img)
        return img, label

train_ds = TransformSubset(full_ds, train_idx, train_transforms)
val_ds   = TransformSubset(full_ds, val_idx,   val_test_transforms)
test_ds  = TransformSubset(full_ds, test_idx,  val_test_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# Cell 5 – Build EfficientNet-B0 with configurable fine-tuning depth

def build_model(experiment: str) -> nn.Module:
    """
    experiment:
      'feature_extraction'  – freeze entire backbone, train classifier only
      'last_block'          – freeze all except last block + classifier
      'partial_finetune'    – unfreeze last 2-3 blocks + classifier
      'full_finetune'       – entire network trainable
    """
    model = torchvision.models.efficientnet_b0(
        weights=torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1
    )

    # --- Replace classifier head for 37 classes ---
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features, NUM_CLASSES),
    )

    # --- Freeze / unfreeze by experiment ---
    # First freeze everything
    for param in model.parameters():
        param.requires_grad = False

    # Always unfreeze new classifier head
    for param in model.classifier.parameters():
        param.requires_grad = True

    if experiment == 'feature_extraction':
        pass  # backbone fully frozen

    elif experiment == 'last_block':
        # Unfreeze only the last MBConv block (features[-1])
        for param in model.features[-1].parameters():
            param.requires_grad = True

    elif experiment == 'partial_finetune':
        # Unfreeze last 2-3 blocks (features[-3:])
        for param in model.features[-3:].parameters():
            param.requires_grad = True

    elif experiment == 'full_finetune':
        for param in model.parameters():
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'[{experiment}] Trainable params: {trainable:,} / {total:,}')

    return model.to(device)

#
exp_name = 'feature_extraction'

model = build_model(exp_name)

model_path = os.path.join(
    SAVE_DIR,
    f'{exp_name}_best.pth'
)

checkpoint = torch.load(
    model_path,
    map_location=device
)

model.load_state_dict(checkpoint['model_state_dict'])

model.eval()

print(f"Loaded model: {exp_name}")
print(f"Test Accuracy: {checkpoint['test_accuracy']:.4f}")
print(f"Test F1 Score: {checkpoint['test_f1']:.4f}")

In [ ]:
# Cell 6 – Training and evaluation functions

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out  = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (out.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out  = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * xb.size(0)
            preds = out.argmax(1)
            correct    += (preds == yb).sum().item()
            total      += xb.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
    avg_loss = total_loss / total
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, prec, rec, f1, all_preds, all_labels

In [ ]:
# import library and connect to google drive

import os
import torch

model_dir = '/content/drive/MyDrive/model_checkpoints'
print(os.listdir(model_dir))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_paths = {
    'feature_extraction': os.path.join(model_dir, '/content/drive/MyDrive/model_checkpoints/feature_extraction_best.pth'),
    'full_finetune': os.path.join(model_dir, '/content/drive/MyDrive/model_checkpoints/full_finetune_best.pth'),
    'last_block': os.path.join(model_dir, '/content/drive/MyDrive/model_checkpoints/last_block_best.pth'),
    'partial_finetune': os.path.join(model_dir, '/content/drive/MyDrive/model_checkpoints/partial_finetune_best.pth')
}

# Instantiate the models first using the helper function build_model
feature_extraction_model = build_model('feature_extraction')
full_finetune_model = build_model('full_finetune')
last_block_model = build_model('last_block')
partial_finetune_model = build_model('partial_finetune')

models = {
    'feature_extraction': feature_extraction_model,
    'full_finetune': full_finetune_model,
    'last_block': last_block_model,
    'partial_finetune': partial_finetune_model
}

for name, model in models.items():
    path = model_paths[name]

    if os.path.exists(path):
        checkpoint = torch.load(path, map_location=device, weights_only=True)

        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            checkpoint = checkpoint['state_dict']
        elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            checkpoint = checkpoint['model_state_dict']

        model.load_state_dict(checkpoint)
        model.to(device)
        model.eval()

        print(f'{name}: Loaded successfully')
    else:
        print(f'{name}: File not found - {path}')

In [ ]:
# Cell 7 – Run all 4 experiments and collect results

experiments = [
    'feature_extraction',   # Exp 1
    'last_block',           # Exp 2
    'partial_finetune',     # Exp 3
    'full_finetune',        # Exp 4
]

results = {}   # store metrics per experiment

for exp_name in experiments:
    print(f'\n{"="*60}')
    print(f'  EXPERIMENT: {exp_name.upper()}')
    print(f'{"="*60}')

    model     = build_model(exp_name)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc  = 0.0
    best_state    = None
    start_time    = time.time()

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc, vl_prec, vl_rec, vl_f1, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}

        print(f'  Epoch {epoch:02d}/{EPOCHS} | '
              f'Train loss {tr_loss:.4f} acc {tr_acc:.4f} | '
              f'Val loss {vl_loss:.4f} acc {vl_acc:.4f}')

    elapsed = time.time() - start_time

    # Load best checkpoint and evaluate on TEST set
    model.load_state_dict(best_state)
    te_loss, te_acc, te_prec, te_rec, te_f1, te_preds, te_labels = \
        evaluate(model, test_loader, criterion)

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    results[exp_name] = {
        'history'          : history,
        'val_loss'         : min(history['val_loss']),
        'test_acc'         : te_acc,
        'test_precision'   : te_prec,
        'test_recall'      : te_rec,
        'test_f1'          : te_f1,
        'training_time_s'  : elapsed,
        'trainable_params' : trainable_params,
        'preds'            : te_preds,
        'labels'           : te_labels,
    }

    print(f'\n  ✅ Test  → Acc: {te_acc:.4f} | Prec: {te_prec:.4f} | '
          f'Rec: {te_rec:.4f} | F1: {te_f1:.4f}')
    print(f'  ⏱  Training time: {elapsed/60:.2f} min | '
          f'Trainable params: {trainable_params:,}')



In [ ]:
# Cell 8 – Results summary table
rows = []
labels_map = {
    'feature_extraction': 'Exp 1: Feature Extraction',
    'last_block'        : 'Exp 2: Last Block',
    'partial_finetune'  : 'Exp 3: Partial Fine-Tune',
    'full_finetune'     : 'Exp 4: Full Fine-Tune',
}
for exp, r in results.items():
    rows.append({
        'Experiment'         : labels_map[exp],
        'Acc'                : f"{r['test_acc']:.4f}",
        'Precision'          : f"{r['test_precision']:.4f}",
        'Recall'             : f"{r['test_recall']:.4f}",
        'F1'                 : f"{r['test_f1']:.4f}",
        'Val Loss'           : f"{r['val_loss']:.4f}",
        'Train Time (min)'   : f"{r['training_time_s']/60:.2f}",
        'Trainable Params'   : f"{r['trainable_params']:,}",
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

In [ ]:
# Cell 9 – Plot 1: Accuracy vs Fine-Tuning Depth
exp_labels = [labels_map[e] for e in experiments]
accs = [results[e]['test_acc'] for e in experiments]

plt.figure(figsize=(8, 5))
bars = plt.bar(exp_labels, accs, color=['#4C72B0','#55A868','#C44E52','#8172B2'])
plt.ylim(0, 1.0)
plt.ylabel('Test Accuracy')
plt.title('Accuracy vs Fine-Tuning Depth')
plt.xticks(rotation=15, ha='right')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 – Plot 2: Loss curves for all experiments
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
for ax, exp, color in zip(axes, experiments, colors):
    h = results[exp]['history']
    ep = range(1, EPOCHS + 1)
    ax.plot(ep, h['train_loss'], label='Train Loss', color=color, linewidth=2)
    ax.plot(ep, h['val_loss'],   label='Val Loss',   color=color, linewidth=2, linestyle='--')
    ax.set_title(labels_map[exp])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Training / Validation Loss Curves', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 – Plot 3: Trainable Parameters vs Fine-Tuning Depth
params = [results[e]['trainable_params'] for e in experiments]

plt.figure(figsize=(8, 5))
bars = plt.bar(exp_labels, params, color=['#4C72B0','#55A868','#C44E52','#8172B2'])
plt.ylabel('Trainable Parameters')
plt.title('Trainable Parameters vs Fine-Tuning Depth')
plt.xticks(rotation=15, ha='right')
plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for bar, p in zip(bars, params):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f'{p/1e6:.2f}M', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 – Plot 4: Training Time vs Fine-Tuning Depth
times_min = [results[e]['training_time_s'] / 60 for e in experiments]

plt.figure(figsize=(8, 5))
bars = plt.bar(exp_labels, times_min, color=['#4C72B0','#55A868','#C44E52','#8172B2'])
plt.ylabel('Training Time (minutes)')
plt.title('Training Time vs Fine-Tuning Depth')
plt.xticks(rotation=15, ha='right')
for bar, t in zip(bars, times_min):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{t:.1f}m', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 – Plot 5: Confusion Matrix for the best experiment
best_exp = max(results, key=lambda e: results[e]['test_acc'])
print(f'Best experiment: {labels_map[best_exp]} (Acc={results[best_exp]["test_acc"]:.4f})')

cm = confusion_matrix(results[best_exp]['labels'], results[best_exp]['preds'])

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=range(NUM_CLASSES),
            yticklabels=range(NUM_CLASSES))
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title(f'Confusion Matrix — {labels_map[best_exp]}')
plt.tight_layout()
plt.show()

# Detailed classification report
print(classification_report(results[best_exp]['labels'],
                             results[best_exp]['preds'],
                             zero_division=0))

In [ ]:
# Cell 14 – Answer the key project questions
print("=" * 65)
print("ANALYSIS: Effect of Fine-Tuning Depth")
print("=" * 65)

best_acc = max(results[e]['test_acc'] for e in experiments)
worst_acc = min(results[e]['test_acc'] for e in experiments)

print(f"\n1. Accuracy range across experiments: {worst_acc:.4f} → {best_acc:.4f}")
print(f"2. Best strategy:  {labels_map[max(results, key=lambda e: results[e]['test_acc'])]}")
print(f"3. Fastest strategy: {labels_map[min(results, key=lambda e: results[e]['training_time_s'])]}")
print(f"4. Fewest params:  {labels_map[min(results, key=lambda e: results[e]['trainable_params'])]}")

print("\nKey Observations:")
print("  - Feature Extraction (frozen backbone) = lowest accuracy but fastest.")
print("  - Full Fine-Tuning = most trainable params and longest training time.")
print("  - Does more fine-tuning always help? Compare Exp 3 vs Exp 4 accuracy.")
print("  - For small datasets, full fine-tuning may cause overfitting")
print("    (watch for val_loss increasing while train_loss falls).")
print("  - Best trade-off is often Partial Fine-Tuning (Exp 3).")

**Model Evaluation**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
import torch

# Upload JPG image
uploaded = files.upload()

if len(uploaded) > 0:
    # Read image
    img_path = list(uploaded.keys())[0]
    img = Image.open(img_path).convert("RGB")

    # Display image
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    # Read image
    # Preprocess image using the validation PyTorch transforms
    # This handles scaling, resizing, and normalizing correctly
    input_tensor = val_test_transforms(img)
    input_batch = input_tensor.unsqueeze(0).to(device) # Add batch dimension and send to GPU/CPU

    # Prediction in PyTorch
    model.eval()
    with torch.no_grad():
        output = model(input_batch)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        predicted_class_idx = torch.argmax(probabilities).item()
        confidence = probabilities[predicted_class_idx].item() * 100

    # Map 37 breed indices back to their names
    breed_name = full_trainval.classes[predicted_class_idx]

    # Identify if the breed is a cat or a dog
    # In the Oxford-IIIT Pet dataset, class names that are cats start with an uppercase letter,
    # while dogs start with a lowercase letter in raw format. Since standard representations
    # can vary, we check the dataset metadata directly. To be absolutely precise:
    # Indices of cats in Oxford-IIIT Pet dataset are: 0, 5, 6, 7, 9, 11, 20, 23, 26, 27, 31, 32
    cat_indices = {0, 5, 6, 7, 9, 11, 20, 23, 26, 27, 31, 32}
    animal_type = "Cat" if predicted_class_idx in cat_indices else "Dog"

    print(f"Predicted Breed: {breed_name}")
    print(f"Predicted Animal Type: {animal_type}")
    print(f"Confidence: {confidence:.2f}%")
else:
    print("No image was uploaded.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files
import torch

# Upload JPG image
uploaded = files.upload()

if len(uploaded) > 0:
    # Read image
    img_path = list(uploaded.keys())[0]
    img = Image.open(img_path).convert("RGB")

    # Display image
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    # Read image
    # Preprocess image using the validation PyTorch transforms
    # This handles scaling, resizing, and normalizing correctly
    input_tensor = val_test_transforms(img)
    input_batch = input_tensor.unsqueeze(0).to(device) # Add batch dimension and send to GPU/CPU

    # Prediction in PyTorch
    model.eval()
    with torch.no_grad():
        output = model(input_batch)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        predicted_class_idx = torch.argmax(probabilities).item()
        confidence = probabilities[predicted_class_idx].item() * 100

    # Map 37 breed indices back to their names
    breed_name = full_trainval.classes[predicted_class_idx]

    # Identify if the breed is a cat or a dog
    # In the Oxford-IIIT Pet dataset, class names that are cats start with an uppercase letter,
    # while dogs start with a lowercase letter in raw format. Since standard representations
    # can vary, we check the dataset metadata directly. To be absolutely precise:
    # Indices of cats in Oxford-IIIT Pet dataset are: 0, 5, 6, 7, 9, 11, 20, 23, 26, 27, 31, 32
    cat_indices = {0, 5, 6, 7, 9, 11, 20, 23, 26, 27, 31, 32}
    animal_type = "Cat" if predicted_class_idx in cat_indices else "Dog"

    print(f"Predicted Breed: {breed_name}")
    print(f"Predicted Animal Type: {animal_type}")
    print(f"Confidence: {confidence:.2f}%")
else:
    print("No image was uploaded.")